<a href="https://colab.research.google.com/github/eeeewyz/RAG/blob/main/2_retrieval%20metrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ungraded Lab -  Retrieval Metrics
---

In this lab, you will be working on retrieving and analyzing metrics for a RAG system. RAG models are designed to improve the quality of generated responses by retrieving relevant documents from a knowledge base. Your goal is to evaluate the retrieval component by calculating precision and recall metrics, along with context precision and context recall.

In this lab, you will learn:
- How to compute precision and recall metrics
- How to apply these metrics in information retrieval
- How to work with a concrete dataset to test the retrieval capabilities of semantic-based searches

You will be using the `sentence-transformers` library to convert text to embeddings, allowing efficient similarity computations. To compute retrieval metrics, you need a labeled dataset.

---
<h4 style="color:black; font-weight:bold;">USING THE TABLE OF CONTENTS</h4>

JupyterLab provides an easy way for you to navigate through your assignment. It's located under the Table of Contents tab, found in the left panel, as shown in the picture below.

![TOC Location](images/toc.png)


# Table of Contents
- [ 1 - The dataset](#1)
  - [ 1.1 Preprocessing and Vectorizing Data](#1-1)
  - [ 1.2 Basic functions for retrieve](#1-2)
- [ 2 - Retrieving metric](#2)
  - [ 2.1 Precision](#2-1)
  - [ 2.2 Recall](#2-2)
  - [ 2.3 Computing metrics over some queries](#2-3)


## 1 - Introduction
---

Retrieval metrics are fundamental in RAG systems, as they provide a way to measure performance. To effectively gauge performance, you need a labeled dataset—one where the answers to specific queries are known—allowing you to compare these results with those generated by your RAG system. In this lab, you will use a pre-labeled dataset and focus on Precision and Recall metrics.

<div style="text-align: center;">
  <img src="images/precision_recall.png" alt="Description" style="width: 70%;">
</div>

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np
import matplotlib.pyplot as plt
import joblib
import os

<a id='1'></a>
### 1.1 The dataset

The [20 Newsgroups dataset](https://scikit-learn.org/0.19/datasets/twenty_newsgroups.html) is a classic text dataset with text data on various topics, with labeled categories. Let's use the `sklearn.datasets` module to load this dataset.

In [ ]:
import pandas as pd

# 导入 20 Newsgroups 数据集加载工具
from sklearn.datasets import fetch_20newsgroups


# 加载训练集
newsgroups_train = fetch_20newsgroups(
    subset='train',       # 加载训练数据
    shuffle=True,         # 打乱数据顺序
    random_state=42,      # 固定随机结果
    data_home='./dataset' # 数据保存位置
)


# 将文本和类别编号组成一个表格
df = pd.DataFrame({
    # 每一篇新闻的文本内容
    'text': newsgroups_train.data,

    # 每一篇新闻对应的类别编号
    'category': newsgroups_train.target
})


# 查看前 5 条数据
print(df.head())


# 查看表格大小：(数据条数, 列数)
print("\nDataset Size:", df.shape)


# 查看类别数量
print(
    "\nNumber of Categories:",
    len(newsgroups_train.target_names)
)


# 查看所有类别名称
print(
    "\nCategories:",
    newsgroups_train.target_names
)

                                                text  category
0  From: lerxst@wam.umd.edu (where's my thing)\nS...         7
1  From: guykuo@carson.u.washington.edu (Guy Kuo)...         4
2  From: twillis@ec.ecn.purdue.edu (Thomas E Will...         4
3  From: jgreen@amber (Joe Green)\nSubject: Re: W...         1
4  From: jcm@head-cfa.harvard.edu (Jonathan McDow...        14

Dataset Size: (11314, 2)

Number of Categories: 20

Categories: ['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']


In [ ]:
print(f"TEXT:\n\t{df['text'][0]}\nCATEGORY:\n\t{newsgroups_train.target_names[df['category'][0]]}")

TEXT:
	From: lerxst@wam.umd.edu (where's my thing)
Subject: WHAT car is this!?
Nntp-Posting-Host: rac3.wam.umd.edu
Organization: University of Maryland, College Park
Lines: 15

 I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.

Thanks,
- IL
   ---- brought to you by your neighborhood Lerxst ----





CATEGORY:
	rec.autos


<a id='1-1'></a>
### 1.1 Preprocessing and Vectorizing Data

In this section, you'll preprocess the text data by cleaning it and then vectorize the text using a pre-trained model from the `sentence-transformers` library. You will use the model `BAAI/bge-base-en-v1.5` for encoding the sentences into vectors. To save time, the dataset has been embedded ahead of time for you, so the model will be used only to vectorize the prompts.

In [ ]:
# 指定要使用的预训练文本向量模型
# bge-base-en-v1.5：具体模型名称
# 这个模型的作用是把英文文本转换成 Embedding 向量
model_name = "BAAI/bge-base-en-v1.5"


# 加载本地已经下载好的 Sentence Transformer 模型
model = SentenceTransformer(
    # 例如：
    # "/home/models"
    # +
    # "BAAI/bge-base-en-v1.5"
    # ↓
    # "/home/models/BAAI/bge-base-en-v1.5"
    os.path.join(
        os.environ['MODEL_PATH'],
        model_name
    )
)

# 从 embeddings.joblib 文件中读取之前保存好的文本向量
# joblib：
# 一个用于保存和读取 Python 对象的工具库
#
# load：
# 表示“读取、加载”
#
# 'embeddings.joblib'：表示文件路径
# 要读取的文件名，里面通常保存了所有新闻文本的向量
embedding_vectors = joblib.load('embeddings.joblib')

embeddings.joblib
[
    [0.12, -0.04, 0.31, ...],
    [0.08,  0.17, -0.21, ...],
    [-0.11, 0.23, 0.09, ...]
]

In [ ]:
type(embedding_vectors)

list

In [ ]:
len(embedding_vectors)

11314

<a id='1-2'></a>
### 1.2 Basic functions for retrieval

Now let's implement a basic RAG mechanism by performing a similarity search over our precomputed embeddings. This code uses cosine similarity to find the most relevant documents for a given query. Let's first define our basic functions.


| 方法                     | 作用                  | 简单理解                     |
| ---------------------- | ------------------- | ------------------------ |
| `hasattr(obj, "name")` | 判断对象有没有某个属性或方法      | 检查“能不能用这个功能”             |
| `np.asarray()`         | 转成 NumPy 数组         | 统一数据格式                   |
| `.ravel()`             | 把数组展开成一维            | 变成一条向量                   |
| `.ndim`                | 查看数组有几个维度           | 判断是一维还是二维                |
| `np.atleast_2d()`      | 保证数组至少是二维           | `[1,2,3]` 变成 `[[1,2,3]]` |
| `np.errstate()`        | 临时忽略 NumPy 计算警告     | 防止除零警告刷出来                |
| `np.linalg.norm()`     | 计算向量长度              | 计算余弦相似度里的模长              |
| `.tolist()`            | NumPy 数组转 Python 列表 | `ndarray` 变成 `list`      |


In [ ]:
def preprocess_text(text):
    """
    Preprocess the text data by removing leading and trailing whitespace.

    Parameters:
    text (str): The input text to preprocess.

    Returns:
    str: The preprocessed text, with leading and trailing whitespace removed.
    """
    # Example preprocessing: remove leading/trailing whitespace
    text = text.strip()
    return text


def cosine_similarity(v1, array_of_vectors):
    """
    计算余弦相似度。

    v1：
        一个查询向量，例如用户问题的 embedding。

    array_of_vectors：
        可以是一个向量，也可以是多个向量组成的二维数组，
        例如所有新闻文本的 embedding。

    返回值：
        如果输入一个向量，返回一个相似度 float。
        如果输入多个向量，返回一个相似度列表。
    """

    # 如果 v1 是 PyTorch Tensor：
    # detach()：从计算图中分离
    # cpu()：移动到 CPU
    # numpy()：转换成 NumPy 数组
    if hasattr(v1, "detach"):
        v1 = v1.detach().cpu().numpy()

    # 转换为 float32 类型的 NumPy 数组
    # ravel() 将向量变成一维形式
    v1 = np.asarray(v1, dtype=np.float32).ravel()

    # 如果 array_of_vectors 是 PyTorch Tensor，
    # 同样转换成 NumPy 数组
    if hasattr(array_of_vectors, "detach"):
        array_of_vectors = array_of_vectors.detach().cpu().numpy()

    # A 表示待比较的一个或多个向量
    A = np.asarray(array_of_vectors, dtype=np.float32)

    # 情况一：A 是一维，说明只比较两个向量
    if A.ndim == 1:
        A = A.ravel()

        # 计算余弦相似度公式中的分母：
        # ||v1|| × ||A||
        denom = np.linalg.norm(v1) * np.linalg.norm(A)

        # 如果某个向量长度为 0，避免除以 0，直接返回 0
        # 否则计算：
        # v1 · A / (||v1|| × ||A||)
        return float(
            0.0 if denom == 0
            else np.dot(v1, A) / denom
        )

    # 情况二：A 是二维，表示一次比较多个向量
    # 每一行代表一个 embedding 向量
    A = np.atleast_2d(A)

    # 查询向量 v1 的长度
    v1_norm = np.linalg.norm(v1)

    # axis=1：分别计算 A 中每一行向量的长度
    A_norms = np.linalg.norm(A, axis=1)

    # 计算每个向量对应的分母
    denom = v1_norm * A_norms

    # 暂时忽略除以 0 和无效计算的警告
    with np.errstate(divide='ignore', invalid='ignore'):

        # A @ v1：
        # A 中每一行分别与 v1 进行点积
        #
        # np.where：
        # 如果分母为 0，临时使用 1，防止报错
        sims = (A @ v1) / np.where(
            denom == 0,
            1.0,
            denom
        )

    # 分母为 0 的相似度设置为 0
    sims[denom == 0] = 0.0

    # 将 NumPy 数组转换成 Python list
    return sims.tolist()


def top_k_greatest_indices(lst, k):
    """
    找出列表中数值最大的前 k 个元素所对应的原始索引。

    lst：
        相似度分数列表。

    k：
        需要返回几个结果。

    返回：
        相似度最高的前 k 个元素的索引。
    """

    # enumerate(lst) 会同时得到：
    # 元素原来的索引和元素值
    #
    # 例如：
    # [0.3, 0.8, 0.5]
    # 转换为：
    # [(0, 0.3), (1, 0.8), (2, 0.5)]
    indexed_list = list(enumerate(lst))

    # 根据每一项的第二个元素 x[1]，也就是相似度分数排序
    # reverse=True 表示从大到小排列
    sorted_by_value = sorted(
        indexed_list,
        key=lambda x: x[1],
        reverse=True
    )

    # 取排序后的前 k 项
    # 只保留它们原来的索引
    top_k_indices = [
        index
        for index, value in sorted_by_value[:k]
    ]

    # 返回 Top K 文本的索引
    return top_k_indices

Now let's define the retriever function.

| 写法                                           | 含义                           | 简单理解                    |
| -------------------------------------------- | ---------------------------- | ----------------------- |
| `df.iloc[idx]`                               | 按行号获取 DataFrame 中的数据         | 取第 `idx` 行              |
| `df.iloc[idx]['text']`                       | 获取第 `idx` 行的 `text` 列        | 取对应文档内容                 |
| `df.iloc[idx]['category']`                   | 获取第 `idx` 行的 `category` 列    | 取文档类别编号                 |
| `newsgroups_train.target_names`              | 保存所有类别名称的列表                  | 类别编号对应的名字表              |
| `newsgroups_train.target_names[category_id]` | 根据类别编号获取类别名称                 | 例如 `14 → sci.space`     |
| `x.detach()`                                 | 将 Tensor 从计算图中分离             | 后续不再计算梯度                |
| `x.cpu()`                                    | 把 Tensor 移动到 CPU             | 方便转换成 NumPy             |
| `x.numpy()`                                  | 把 PyTorch Tensor 转成 NumPy 数组 | Tensor → ndarray        |
| `x.detach().cpu().numpy()`                   | 分离计算图、移到 CPU、转成 NumPy        | PyTorch 数据转 NumPy 的常见写法 |


In [ ]:
def retrieve_documents(query, embeddings, model, top_k=5):
    """
    根据查询文本，检索最相似的前 top_k 篇文档。

    query:
        用户输入的查询文本

    embeddings:
        所有文档对应的 embedding 向量

    model:
        用来生成查询向量的 SentenceTransformer 模型

    top_k:
        返回相似度最高的文档数量，默认是 5
    """

    # 1. 清理查询文本，例如删除首尾空格
    query_clean = preprocess_text(query)

    # 2. 把查询文本转换成 embedding 向量
    # convert_to_tensor=False 表示返回 NumPy 数组
    # astype(np.float32) 把数据类型统一为 float32
    query_embedding = model.encode(
        query_clean,
        convert_to_tensor=False
    ).astype(np.float32)

    # 用来保存查询向量和每篇文档向量的相似度
    cosine_scores = []

    # 3. 逐个读取所有文档 embedding
    for x in embeddings:

        # 如果 x 是 PyTorch Tensor，就转换成 NumPy 数组
        if hasattr(x, "detach"):
            x = x.detach().cpu().numpy()

        # 保证 x 是 float32 类型的 NumPy 数组
        x = np.asarray(x, dtype=np.float32)

        # 4. 计算查询向量和当前文档向量的余弦相似度
        score = cosine_similarity(query_embedding, x)

        # 把相似度转换成 float，并保存到列表中
        cosine_scores.append(float(score))

    # 5. 找到相似度最高的前 top_k 个文档索引
    top_results = top_k_greatest_indices(
        cosine_scores,
        k=top_k
    )

    # 输出原始查询文本
    print(f"Query: {query}")

    # 6. 根据索引输出 Top K 文档
    for idx in top_results:

        # df.iloc[idx]：根据行号获取对应文档
        # ['text']：获取文档文本
        # [:200]：只显示前 200 个字符
        print(
            f"Document: {df.iloc[idx]['text'][:200]}..."
        )

        # 获取文档的数字类别
        category_id = df.iloc[idx]['category']

        # 根据类别编号获取类别名称
        category_name = newsgroups_train.target_names[
            category_id
        ]

        print(f"Category: {category_name}...")

        # 每篇结果之间换行
        print("\n\n")

<a id='2'></a>
## 2 - Retrieving metrics

---

Let's explore briefly the most common metrics for retrieval systems: Precision@K and Recall@K.

<a id='2-1'></a>
### 2.1 Precision@K

Precision@K provides an evaluation of the relevancy of the top K retrieved documents. It's calculated as the ratio of relevant documents in the top K results to K (the total number of documents retrieved).

$$\text{Precision@K} = \frac{\text{Number of Relevant Documents in Top K}}{\text{K}}$$

where K is the number of documents retrieved.

In [ ]:
def precision_at_k(relevant_count, k):
    """
    Calculate the Precision@K for a retrieval system.

    Precision@K is the ratio of relevant documents in the top K retrieved documents
    to the total number K of documents retrieved.

    Args:
        relevant_count (int): Number of relevant documents in the top K results.
        k (int): Total number of documents retrieved (K).

    Returns:
        float: The Precision@K value, or 0.0 if k is zero.

    Raises:
        ValueError: If any input is negative.
    """
    if relevant_count < 0 or k < 0:
        raise ValueError("All input values must be non-negative.")

    if k == 0:
        return 0.0

    return relevant_count / k

<a id='2-2'></a>
### 2.2 Recall@K

Recall@K evaluates the retrieval system's ability to find all relevant documents from the dataset within the top K results. It's calculated as the ratio of relevant documents in the top K results to the total number of relevant documents in the entire corpus.

$$\text{Recall@K} = \frac{\text{Number of Relevant Documents in Top K}}{\text{Total Number of Relevant Documents in Corpus}}$$

In [ ]:
def recall_at_k(relevant_count, total_relevant):
    """
    Calculate the Recall@K for a retrieval system.

    Recall@K is the ratio of relevant documents in the top K retrieved documents
    to the total number of relevant documents in the entire corpus.

    Args:
        relevant_count (int): Number of relevant documents in the top K results.
        total_relevant (int): Total number of relevant documents in the corpus.

    Returns:
        float: The Recall@K value, or 0.0 if total_relevant is zero.

    Raises:
        ValueError: If any input is negative.
    """
    if relevant_count < 0 or total_relevant < 0:
        raise ValueError("All input values must be non-negative.")

    if total_relevant == 0:
        return 0.0

    return relevant_count / total_relevant

<a id='2-3'></a>
### 2.3 Computing metrics over some queries

Now let's compute these metrics on some pre-defined queries.

In [ ]:
# Define more complex test queries with their corresponding desired categories
test_queries = [
    {"query": "advancements in space exploration technology", "desired_category": "sci.space"},
    {"query": "real-time rendering techniques in computer graphics", "desired_category": "comp.graphics"},
    {"query": "latest findings in cardiovascular medical research", "desired_category": "sci.med"},
    {"query": "NHL playoffs and team performance statistics", "desired_category": "rec.sport.hockey"},
    {"query": "impacts of cryptography in online security", "desired_category": "sci.crypt"},
    {"query": "the role of electronics in modern computing devices", "desired_category": "sci.electronics"},
    {"query": "motorcycles maintenance tips for enthusiasts", "desired_category": "rec.motorcycles"},
    {"query": "high-performance baseball tactics for championships", "desired_category": "rec.sport.baseball"},
    {"query": "historical influence of politics on society", "desired_category": "talk.politics.misc"},
    {"query": "latest technology trends in the Windows operating system", "desired_category": "comp.os.ms-windows.misc"}

]


In [ ]:
np_embeddings = [
    np.array([0.1, 0.2, 0.3, 0.4]),  # 文档 0
    np.array([0.5, 0.6, 0.7, 0.8]),  # 文档 1
    np.array([0.9, 1.0, 1.1, 1.2])   # 文档 2
]

执行：

E = np.vstack(np_embeddings)

得到：

E = np.array([
    [0.1, 0.2, 0.3, 0.4],
    [0.5, 0.6, 0.7, 0.8],
    [0.9, 1.0, 1.1, 1.2]
])

它的形状：

E.shape
# (3, 4)

含义是：

3 = 文档数量 N
4 = 每个 embedding 的维度 D

也就是：

行	表示什么
E[0]	第 1 篇文档的 embedding
E[1]	第 2 篇文档的 embedding
E[2]	第 3 篇文档的 embedding

In [ ]:
desired_category = item["desired_category"]

这里的 desired_category 通常不是列表，而是一个字符串标签。

例如：

queries = [
    {
        "query": "space exploration",
        "desired_category": "sci.space"
    },
    {
        "query": "computer graphics",
        "desired_category": "comp.graphics"
    }
]

In [ ]:
def compute_metrics(queries, embeddings, model, top_k=5):
    """
    对多条查询计算 Precision@K 和 Recall@K。

    queries:
        查询列表，每一项包含 query 和 desired_category

    embeddings:
        所有文档的 embedding

    model:
        生成查询 embedding 的模型

    top_k:
        每次取相似度最高的前 K 篇文档
    """

    # 保存每条查询的评估结果
    results = []

    # 把所有文档 embedding 统一转换成 NumPy
    np_embeddings = []

    for x in embeddings:

        # 如果 x 是 PyTorch Tensor，就转成 NumPy
        if hasattr(x, "detach"):
            x = x.detach().cpu().numpy()

        # 转成 float32 的一维 NumPy 向量
        x = np.asarray(x, dtype=np.float32).ravel()

        # 保存到列表
        np_embeddings.append(x)

    # 把多个一维向量按行堆叠成二维矩阵
    # E 的形状为：(文档数量 N, 向量维度 D)
    E = np.vstack(np_embeddings)

    # 逐条处理查询
    for item in queries:

        # 查询文本
        query = item["query"]

        # 该查询期望检索到的正确类别
        desired_category = item["desired_category"]

        # 清理查询文本
        q_clean = preprocess_text(query)

        # 把查询文本转换成 embedding
        # convert_to_tensor=False：返回 NumPy 数组
        q_emb = model.encode(
            q_clean,
            convert_to_tensor=False
        )

        # 统一为 float32 的一维向量
        q_emb = np.asarray(
            q_emb,
            dtype=np.float32
        ).ravel()

        # 一次性计算查询向量和所有文档向量的余弦相似度
        cosine_scores = cosine_similarity(q_emb, E)

        # 找出相似度最高的前 top_k 个文档索引
        top_results = top_k_greatest_indices(
            cosine_scores,
            k=top_k
        )

        # 根据文档索引获取 Top K 文档的类别名称
        retrieved_categories = [
            newsgroups_train.target_names[
                df.iloc[idx]["category"]
            ]
            for idx in top_results
        ]

        # 统计 Top K 中有多少文档属于正确类别
        relevant_in_top_k = sum(
            1
            for cat in retrieved_categories
            if cat == desired_category
        )

         # 统计整个数据集中属于正确类别的文档总数
        total_relevant_in_corpus = sum(
            1
            # 遍历 df 中所有文档的行号：0, 1, 2, ...
            for idx in range(len(df))

            # df.iloc[idx]["category"]：
            # 先获取第 idx 行文档，再取出它的类别编号
            #
            # newsgroups_train.target_names[...]：
            # 根据类别编号获取真正的类别名称
            #
            # 如果当前文档的类别名称等于 desired_category，
            # 就产生一个数字 1
            if newsgroups_train.target_names[
                df.iloc[idx]["category"]
            ] == desired_category
        )

        # sum() 会把所有产生的 1 加起来
        # 最终结果表示：
        # 整个数据集中一共有多少篇文档属于 desired_category

        # 计算 Precision@K
        p = precision_at_k(
            relevant_in_top_k,
            top_k
        )

        # 计算 Recall@K
        r = recall_at_k(
            relevant_in_top_k,
            total_relevant_in_corpus
        )

        # 保存当前查询的评估结果
        results.append({
            "query": query,
            "precision@k": p,
            "recall@k": r,
        })

    # 返回所有查询的结果
    return results

In [ ]:
# Run the queries and compute metrics with different K values
k_values = [5, 20, 50]

for k in k_values:
    print(f"\n{'='*80}")
    print(f"Results with K={k}:")
    print('='*80)
    results = compute_metrics(test_queries, embedding_vectors, model, top_k=k)

    # Display the results
    for result in results:
        print(f"Query: {result['query']}")
        print(f"  Precision@{k}: {result['precision@k']:.2f}, Recall@{k}: {result['recall@k']:.2f}")
        print()


Results with K=5:
Query: advancements in space exploration technology
  Precision@5: 1.00, Recall@5: 0.01

Query: real-time rendering techniques in computer graphics
  Precision@5: 1.00, Recall@5: 0.01

Query: latest findings in cardiovascular medical research
  Precision@5: 1.00, Recall@5: 0.01

Query: NHL playoffs and team performance statistics
  Precision@5: 1.00, Recall@5: 0.01

Query: impacts of cryptography in online security
  Precision@5: 1.00, Recall@5: 0.01

Query: the role of electronics in modern computing devices
  Precision@5: 1.00, Recall@5: 0.01

Query: motorcycles maintenance tips for enthusiasts
  Precision@5: 1.00, Recall@5: 0.01

Query: high-performance baseball tactics for championships
  Precision@5: 1.00, Recall@5: 0.01

Query: historical influence of politics on society
  Precision@5: 0.40, Recall@5: 0.00

Query: latest technology trends in the Windows operating system
  Precision@5: 0.80, Recall@5: 0.01


Results with K=20:
Query: advancements in space explor

**Understanding the Results:**

The results above clearly demonstrate the **precision-recall tradeoff** in retrieval systems as we vary K from 5 to 20 to 50:

**Precision@K Trends (generally decreases as K increases):**

- **At K=5**: Most queries achieve very high precision (0.80-1.00), with 8 out of 10 queries having perfect precision (1.00). This means nearly all retrieved documents are highly relevant.
  
- **At K=20**: Precision starts to decline for some queries:
  - "electronics in computing devices" drops to 0.80 (from 1.00)
  - "Windows operating system" drops to 0.65 (from 0.80)
  - "motorcycles maintenance" drops to 0.95 (from 1.00)
  
- **At K=50**: Precision decreases further as we retrieve more documents:
  - "computer graphics" drops to 0.88 (from 1.00)
  - "electronics in computing devices" drops to 0.66 (from 0.80)
  - "Windows operating system" drops to 0.60 (from 0.65)
  - "historical influence of politics" remains around 0.50-0.52 (the lowest across all K values)

**Recall@K Trends (increases as K increases):**

- **At K=5**: Recall is very low (~0.01 or 1%), meaning only about 1% of all relevant documents are retrieved
  
- **At K=20**: Recall triples to ~0.03 (3%), capturing more relevant documents
  
- **At K=50**: Recall increases to 0.05-0.08 (5-8%), capturing approximately 8 times more relevant documents than K=5

**Key Observations:**

1. **The tradeoff is clear**: As K increases, we retrieve more of the total relevant documents (higher recall), but at the cost of including some irrelevant documents (lower precision).

2. **Some queries are harder than others**: The query "historical influence of politics on society" consistently shows the lowest precision (0.40-0.52), suggesting that this query is semantically ambiguous or the category "talk.politics.misc" is harder to distinguish from related categories.

3. **For RAG systems, K=5 to K=20 is often optimal**: These values provide high precision (most retrieved documents are relevant) while keeping the context size manageable for the LLM. Even though recall is low, the goal is to find the *most relevant* documents, not *all* relevant documents.

4. **Recall remains relatively low even at K=50**: This is expected since each category contains hundreds of documents (500-600), so retrieving 50 documents only captures ~8-10% of the total relevant documents. To achieve high recall, we would need K values in the hundreds, which would severely impact precision and be impractical for RAG applications.

Congratulations on finishing this Ungraded Lab! Keep it up!